In [21]:
# Step 1: Imports
import cv2
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import torch
from collections import deque, Counter

In [22]:
# Step 2: Configuration
VIDEO_PATH = "raw_data/videos/76_RC_PTZ 1_20260427_210000_20260427_211500_4.mp4"
XML_OUT = "Runs/trajectories/pedestrian_annotations.xml"
CSV_OUT = "Runs/trajectories/pedestrian_trajectories.csv"
MODEL_NAME = "yolo11n.pt"  # High accuracy for pedestrians
CONF_THRESHOLD = 0.4
END_FRAME = 200
START_FRAME = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [23]:
# Step 3: Pedestrian-Only Tracking with Rider Filter
def calculate_iou(box1, box2):
    # Simple intersection area check
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    return inter_area / box1_area if box1_area > 0 else 0

def run_pedestrian_pass():
    model = YOLO(MODEL_NAME).to(device)
    cap = cv2.VideoCapture(VIDEO_PATH)
    width, height = int(cap.get(3)), int(cap.get(4))
    data_log = []; ID_OFFSET = 50000

    for frame_idx in tqdm(range(START_FRAME, END_FRAME + 1), desc="Processing"):
        ret, frame = cap.read()
        if not ret: break
        
        # Detect Persons (0), Bicycles (1), and Motorcycles (3)
        results = model.track(frame, persist=True, classes=[0, 1, 3], conf=CONF_THRESHOLD, imgsz=1280, verbose=False)
        
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clss = results[0].boxes.cls.cpu().numpy().astype(int)
            confs = results[0].boxes.conf.cpu().numpy()

            persons = []
            vehicles = []
            for b, tid, c, s in zip(boxes, ids, clss, confs):
                if c == 0: persons.append({"box": b, "id": tid, "score": s})
                else: vehicles.append(b)

            for p in persons:
                # If person overlaps with any bike/moto, we skip them (they are a rider)
                is_rider = any(calculate_iou(p["box"], v) > 0.3 for v in vehicles)
                if not is_rider:
                    data_log.append({
                        "frame": frame_idx, "track_id": p["id"] + ID_OFFSET, "label": "Pedestrian",
                        "confidence": float(p["score"]), "xtl": p["box"][0], "ytl": p["box"][1], 
                        "xbr": p["box"][2], "ybr": p["box"][3]
                    })
    cap.release()
    df = pd.DataFrame(data_log)
    df.to_csv(CSV_OUT, index=False)
    return df, width, height

ped_df, v_w, v_h = run_pedestrian_pass()


Processing: 100%|██████████| 201/201 [00:08<00:00, 24.35it/s]


In [24]:
# Step 4: Build Pedestrian XML
def build_ped_xml(df, out_path, w, h):
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"
    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "size").text = str(END_FRAME + 1)
    ET.SubElement(task, "mode").text = "interpolation"
    
    labels = ET.SubElement(task, "labels")
    l = ET.SubElement(labels, "label")
    ET.SubElement(l, "name").text = "Pedestrian"
    ET.SubElement(l, "type").text = "bbox"
    
    osize = ET.SubElement(task, "original_size")
    ET.SubElement(osize, "width").text = str(w)
    ET.SubElement(osize, "height").text = str(h)

    for tid, track_df in tqdm(df.groupby("track_id"), desc="Writing XML"):
        track_el = ET.SubElement(root, "track", id=str(int(tid)), label="Pedestrian")
        track_df = track_df.sort_values("frame")
        
        for row in track_df.itertuples():
            ET.SubElement(track_el, "box", 
                          frame=str(int(row.frame)), 
                          xtl=f"{row.xtl:.2f}", ytl=f"{row.ytl:.2f}", 
                          xbr=f"{row.xbr:.2f}", ybr=f"{row.ybr:.2f}", 
                          outside="0", keyframe="1")
        
        # Add termination
        last_f = int(track_df["frame"].max())
        if last_f + 1 <= END_FRAME:
            ET.SubElement(track_el, "box", 
                          frame=str(last_f + 1), 
                          xtl=f"{track_df.iloc[-1].xtl:.2f}", ytl=f"{track_df.iloc[-1].ytl:.2f}", 
                          xbr=f"{track_df.iloc[-1].xbr:.2f}", ybr=f"{track_df.iloc[-1].ybr:.2f}", 
                          outside="1", keyframe="1")

    ET.indent(root, space="  ", level=0)
    ET.ElementTree(root).write(out_path, encoding="utf-8", xml_declaration=True)
    print(f"--- Pedestrian XML Saved: {out_path} ---")

build_ped_xml(ped_df, XML_OUT, v_w, v_h)

Writing XML: 100%|██████████| 26/26 [00:00<00:00, 1056.64it/s]

--- Pedestrian XML Saved: Runs/trajectories/pedestrian_annotations.xml ---


In [25]:
# Step 5: Visual Verification (Annotated Pedestrian Video)
def save_verification_video(df, video_path, out_path, start_frame, end_frame):
    cap = cv2.VideoCapture(video_path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    for f_idx in tqdm(range(start_frame, end_frame + 1), desc="Saving Verification Video"):
        ret, frame = cap.read()
        if not ret: break
        
        frame_data = df[df["frame"] == f_idx]
        for row in frame_data.itertuples():
            cv2.rectangle(frame, (int(row.xtl), int(row.ytl)), (int(row.xbr), int(row.ybr)), (255, 0, 255), 2)
            cv2.putText(frame, f"PED ID:{row.track_id}", (int(row.xtl), int(row.ytl)-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2)
        
        writer.write(frame)
    
    cap.release()
    writer.release()
    print(f"--- Verification Video Saved: {out_path} ---")

VERIFY_OUT = "Runs/annotated_videos/pedestrian_verify.mp4"
save_verification_video(ped_df, VIDEO_PATH, VERIFY_OUT, START_FRAME, END_FRAME)

Saving Verification Video: 100%|██████████| 201/201 [00:02<00:00, 81.92it/s]

--- Verification Video Saved: Runs/annotated_videos/pedestrian_verify.mp4 ---
